In [3]:
import duckdb

conn = duckdb.connect('../warehouse/scholarhub.duckdb')

print("=" * 60)
print("DUCKDB VERIFICATION")
print("=" * 60)

# Count records
count = conn.execute("SELECT COUNT(*) FROM raw_nsf_awards").fetchone()[0]
print(f"\n✅ Total records in raw_nsf_awards: {count}")

# Quality scores
quality = conn.execute("""
    SELECT
        AVG(quality_score) as avg_quality,
        MIN(quality_score) as min_quality,
        MAX(quality_score) as max_quality
    FROM raw_nsf_awards
""").fetchone()
print(f"\n📊 Quality Scores:")
print(f"   - Average: {quality[0]:.3f}")
print(f"   - Minimum: {quality[1]:.3f}")
print(f"   - Maximum: {quality[2]:.3f}")

# Sample records
print(f"\n📝 Sample Award Titles:")
titles = conn.execute("""
    SELECT response_json::JSON->>'title' as title
    FROM raw_nsf_awards
    LIMIT 5
""").fetchall()
for i, (title,) in enumerate(titles, 1):
    print(f"   {i}. {title[:80]}...")

# Extraction dates
dates = conn.execute("""
    SELECT
        MIN(extracted_at) as first_extraction,
        MAX(extracted_at) as last_extraction
    FROM raw_nsf_awards
""").fetchone()
print(f"\n⏰ Extraction Timeline:")
print(f"   - First record: {dates[0]}")
print(f"   - Last record:  {dates[1]}")

conn.close()
print("\n" + "=" * 60)

DUCKDB VERIFICATION

✅ Total records in raw_nsf_awards: 500

📊 Quality Scores:
   - Average: 0.999
   - Minimum: 0.800
   - Maximum: 1.000

📝 Sample Award Titles:
   1. Examining the Role of District Science Coordinator Professional Learning in Supp...
   2. Collaborative Research: Quantifying Secondary Electron Emission for Predictive E...
   3. REU Site: Novel Techniques and Applications in Catalysis Research Development an...
   4. CAREER: Enabling Single-Step Additive Manufacturing of Ceramics via Laser-Trigge...
   5. CAREER: Quantum-Inspired Dynamic Control in Composite Metastructures with Long-R...

⏰ Extraction Timeline:
   - First record: 2026-03-20 11:14:55.073793
   - Last record:  2026-03-20 11:16:02.932148



In [6]:
import duckdb

conn = duckdb.connect('../warehouse/scholarhub.duckdb')

print("=" * 70)
print("DBT MARTS VERIFICATION")
print("=" * 70)

# List all schemas
print("\n📁 Schemas:")
schemas = conn.execute("SELECT schema_name FROM information_schema.schemata ORDER BY schema_name").fetchall()
for schema in schemas:
    print(f"   - {schema[0]}")

# List all tables in analytics schemas
print("\n📊 Analytics Tables:")
tables = conn.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema LIKE 'analytics%'
    ORDER BY table_schema, table_name
""").fetchall()
for schema, table in tables:
    print(f"   - {schema}.{table}")

# Verify mart_funding_by_year
print("\n\n📈 mart_funding_by_year (sample):")
result = conn.execute("""
    SELECT award_year, total_awards, total_funding, avg_award_size, distinct_pis, distinct_institutions
    FROM analytics_marts.mart_funding_by_year
    ORDER BY award_year DESC
    LIMIT 5
""").fetchall()
print(f"{'Year':<8} {'Awards':<10} {'Total Funding':<18} {'Avg Award':<15} {'PIs':<8} {'Institutions':<15}")
print("-" * 70)
for year, awards, funding, avg_size, pis, institutions in result:
    print(f"{year:<8} {awards:<10} ${funding:>15,.2f} ${avg_size:>12,.2f} {pis:<8} {institutions:<15}")

# Verify mart_funding_by_institution
print("\n\n🏛️  mart_funding_by_institution (top 5):")
result = conn.execute("""
    SELECT institution, state, total_awards, total_funding, funding_rank
    FROM analytics_marts.mart_funding_by_institution
    ORDER BY total_funding DESC
    LIMIT 5
""").fetchall()
print(f"{'Institution':<45} {'State':<6} {'Awards':<8} {'Total Funding':<18} {'Rank':<6}")
print("-" * 70)
for inst, state, awards, funding, rank in result:
    inst_short = (inst[:42] + '...') if inst and len(inst) > 45 else (inst or 'Unknown')
    print(f"{inst_short:<45} {state or 'N/A':<6} {awards:<8} ${funding:>15,.2f} #{rank:<5}")

# Verify mart_funding_by_field
print("\n\n🔬 mart_funding_by_field (sample - recent years):")
result = conn.execute("""
    SELECT directorate, award_year, total_awards, total_funding, yoy_growth_pct
    FROM analytics_marts.mart_funding_by_field
    WHERE award_year >= 2024
    ORDER BY directorate, award_year DESC
    LIMIT 10
""").fetchall()
print(f"{'Directorate':<15} {'Year':<8} {'Awards':<8} {'Total Funding':<18} {'YoY Growth %':<15}")
print("-" * 70)
for directorate, year, awards, funding, growth in result:
    growth_str = f"{growth:>6.1f}%" if growth is not None else "N/A"
    print(f"{directorate or 'Unknown':<15} {year:<8} {awards:<8} ${funding:>15,.2f} {growth_str:<15}")

conn.close()
print("\n" + "=" * 70)


DBT MARTS VERIFICATION

📁 Schemas:
   - analytics_marts
   - analytics_staging
   - information_schema
   - main
   - main
   - main
   - pg_catalog

📊 Analytics Tables:
   - analytics_marts.mart_funding_by_field
   - analytics_marts.mart_funding_by_institution
   - analytics_marts.mart_funding_by_year
   - analytics_staging.stg_nsf_awards


📈 mart_funding_by_year (sample):
Year     Awards     Total Funding      Avg Award       PIs      Institutions   
----------------------------------------------------------------------
2026     499        $ 238,784,066.00 $  478,525.18 477      249            


🏛️  mart_funding_by_institution (top 5):
Institution                                   State  Awards   Total Funding      Rank  
----------------------------------------------------------------------
University of California-Los Angeles          N/A    18       $   8,301,039.00 #1    
University of California-Berkeley             N/A    7        $   6,278,545.00 #2    
Purdue University     